In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import numpy as np
import torch
import torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from sklearn.svm import SVC, LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
import joblib
import json
from tqdm.notebook import tqdm
import warnings
import sys
from sklearn.metrics import precision_score, recall_score, f1_score

warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Cấu hình đường dẫn
DATASET_ROOT = '/kaggle/input/vietnamese-foods'
IMG_ROOT_DIR = os.path.join(DATASET_ROOT, 'Images')

TRAIN_DIR = os.path.join(IMG_ROOT_DIR, 'Train')
VAL_DIR = os.path.join(IMG_ROOT_DIR, 'Validate')
TEST_DIR = os.path.join(IMG_ROOT_DIR, 'Test')


# Class Dataset & Transform
# DINOv2 yêu cầu ảnh đầu vào 224x224, chuẩn hóa ImageNet
dinov2_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class FoodDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.filepaths = []
        self.labels = []
        
        if os.path.exists(root_dir):
            classes = sorted(os.listdir(root_dir))
            for label in classes:
                class_dir = os.path.join(root_dir, label)
                if os.path.isdir(class_dir):
                    for file in os.listdir(class_dir):
                        if file.lower().endswith(('.jpg', '.jpeg')):
                            self.filepaths.append(os.path.join(class_dir, file))
                            self.labels.append(label)
        
    def __len__(self):
        return len(self.filepaths)
    
    def __getitem__(self, idx):
        img_path = self.filepaths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            
            if self.transform:
                image = self.transform(image)
            return image, self.labels[idx]
        except Exception:
            return torch.zeros((3, 224, 224)), self.labels[idx]

# Tạo DataLoader
# Gộp Train + Validate lại để train
train_ds_raw = FoodDataset(TRAIN_DIR, dinov2_transform)
val_ds_raw = FoodDataset(VAL_DIR, dinov2_transform)
full_train_ds = torch.utils.data.ConcatDataset([train_ds_raw, val_ds_raw])
test_ds = FoodDataset(TEST_DIR, dinov2_transform)

train_loader = DataLoader(full_train_ds, batch_size=64, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

print(f"Số ảnh Train: {len(full_train_ds)}")
print(f"Số ảnh Test:  {len(test_ds)}")

In [ ]:
print(" Đang tải model DINOv2 (ViT-S/14)...")
# Dùng torch.hub load model small (nhẹ mà chính xác)
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
dinov2.eval() # Quan trọng: Chế độ Evaluation

# 2. Hàm trích xuất
def extract_features(loader, name="Dataset"):
    feats = []
    lbls = []
    print(f" Đang xử lý {name}...")
    
    with torch.no_grad(): # Tắt tính toán đạo hàm để tiết kiệm RAM
        for images, labels in tqdm(loader):
            images = images.to(device)
            # Forward pass
            output = dinov2(images)

            feats.append(output.cpu().numpy())
            lbls.extend(labels)
            
    return np.concatenate(feats), np.array(lbls)

# 3. Thực hiện
X_train_feats, y_train_raw = extract_features(train_loader, "Tập Train")
X_test_feats, y_test_raw = extract_features(test_loader, "Tập Test")

print(f"\nKích thước Train Features: {X_train_feats.shape}")
print(f"Kích thước Test Features:  {X_test_feats.shape}")

In [ ]:
# Encode nhãn
le = LabelEncoder()
    
# Fit trên toàn bộ nhãn (Train + Test)
all_labels = np.concatenate([y_train_raw, y_test_raw])
le.fit(all_labels)

y_train_enc = le.transform(y_train_raw)
y_test_enc = le.transform(y_test_raw)

# Tạo và lưu file mapping
label_mapping = {label: int(index) for index, label in enumerate(le.classes_)}
with open('label_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(label_mapping, f, ensure_ascii=False, indent=4)
print("Đã lưu file mapping: label_mapping.json")

# Cấu hình Pipeline và GridSearch
# Định nghĩa Pipeline
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()), 
    ('svm', SVC(
        C=10,               # Tham số cố định
        gamma='scale',      # Tham số cố định
        kernel='rbf',       # Tham số cố định
        probability=True,  
        cache_size=2000,  
        random_state=42
    ))
])

# Huấn luyện
svm_pipeline.fit(X_train_feats, y_train_enc)
print("Huấn luyện hoàn tất.")


# --- 3. DỰ ĐOÁN VÀ ĐÁNH GIÁ ---
print("Đang dự đoán trên tập Test...")
y_pred = svm_pipeline.predict(X_test_feats)

# Chuyển ngược lại tên món ăn
y_test_names = le.inverse_transform(y_test_enc)
y_pred_names = le.inverse_transform(y_pred)

# Tính toán các chỉ số
acc = accuracy_score(y_test_names, y_pred_names)
prec = precision_score(y_test_names, y_pred_names, average='weighted')
rec = recall_score(y_test_names, y_pred_names, average='weighted')
f1 = f1_score(y_test_names, y_pred_names, average='weighted')

print("\n--- KẾT QUẢ ĐÁNH GIÁ TỔNG HỢP ---")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")

print("\n--- CHI TIẾT TỪNG LỚP (CLASSIFICATION REPORT) ---")
print(classification_report(y_test_names, y_pred_names))




# --- 5. LƯU MODEL ---
filename = 'dinov2_svm_best_fixed.pkl'
joblib.dump(svm_pipeline, filename)
print(f"Đã lưu model thành công tại: {filename}")

In [ ]:
# --- 4. VẼ MA TRẬN NHẦM LẪN (CONFUSION MATRIX) DẠNG PHẦN TRĂM ---
import matplotlib.pyplot as plt
import seaborn as sns

# Tính ma trận nhầm lẫn
cm = confusion_matrix(y_test_names, y_pred_names, labels=le.classes_)

# Chuyển đổi sang phần trăm (Normalize theo hàng - True Labels)
cm_percent = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-9)

plt.figure(figsize=(30, 30))
# Dùng màu Oranges hoặc Blues để hiển thị
plt.imshow(cm_percent, interpolation='nearest', cmap=plt.cm.Oranges) 
plt.title("Confusion Matrix (%) - DINOv2 + SVM")
plt.colorbar()

tick_marks = np.arange(len(le.classes_))
plt.xticks(tick_marks, le.classes_, rotation=90, fontsize=12)
plt.yticks(tick_marks, le.classes_, fontsize=12)

# Hiển thị số phần trăm trong từng ô
thresh = cm_percent.max() / 2
for i in range(cm_percent.shape[0]):
    for j in range(cm_percent.shape[1]):
        # Chỉ hiển thị text nếu giá trị > 0.1%
        if cm_percent[i, j] > 0.001:
            plt.text(j, i, f"{cm_percent[i, j]*100:.1f}%",
                     horizontalalignment="center",
                     verticalalignment="center",
                     color="white" if cm_percent[i, j] > thresh else "black",
                     fontsize=10)

plt.xlabel("Predicted Label", fontsize=14)
plt.ylabel("True Label", fontsize=14)
plt.tight_layout()
plt.show()